# Function data loader, validator, tabulator, and plotter

This notebook:
1. Loads the CSV file
2. Flags lines with too many commas / too many columns
3. Cleans the malformed rows using the known input dimension for each function
4. Tabulates the data
5. Draws one 2D line graph per function (`row_num` vs `y`) and marks the maximum `y`
6. Saves each graph as a PNG file


In [1]:

import csv
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA_FILE = Path("data.csv")
OUTPUT_DIR = Path("plots")
OUTPUT_DIR.mkdir(exist_ok=True)

EXPECTED_COLUMNS = ["function", "x1", "x2", "x3", "x4", "x5", "x6", "x7", "x8", "y"]
FUNCTION_DIMS = {
    "function_1": 2,
    "function_2": 2,
    "function_3": 3,
    "function_4": 4,
    "function_5": 4,
    "function_6": 5,
    "function_7": 6,
    "function_8": 8,
}
INITIAL_COUNTS = {
    "function_1": 12,
    "function_2": 12,
    "function_3": 17,
    "function_4": 32,
    "function_5": 22,
    "function_6": 22,
    "function_7": 32,
    "function_8": 42,
}


In [2]:

# Scan raw CSV rows and identify malformed lines.
raw_rows = []
malformed_lines = []
blank_lines = []

with DATA_FILE.open("r", encoding="utf-8", newline="") as f:
    reader = csv.reader(f)
    for line_no, row in enumerate(reader, start=1):
        raw_rows.append((line_no, row))
        if len(row) == 0:
            blank_lines.append(line_no)
        elif len(row) > len(EXPECTED_COLUMNS):
            malformed_lines.append((line_no, len(row), row))

print("Expected column count:", len(EXPECTED_COLUMNS))
print("Blank lines:", blank_lines)
print("Lines with too many commas / too many columns:")
for line_no, col_count, row in malformed_lines:
    print(f"  line {line_no}: {col_count} columns -> {row}")


Expected column count: 10
Blank lines: [2, 178, 187, 196, 205, 214, 223, 232, 241, 245, 249]
Lines with too many commas / too many columns:


In [3]:

# Clean rows.
# Strategy for malformed rows:
# - keep the first token as the function name
# - keep the last non-empty token as y
# - collect the numeric x values between them
# - place those x values into x1..x8 and fill the rest with missing values

cleaned_records = []

for line_no, row in raw_rows:
    if len(row) == 0:
        continue

    # Skip the header row
    if line_no == 1:
        continue

    func = row[0].strip()
    if func not in FUNCTION_DIMS:
        continue

    # Remove surrounding spaces from each token
    tokens = [token.strip() for token in row]

    # Last non-empty token is treated as y
    non_empty_positions = [i for i, token in enumerate(tokens) if token != ""]
    if len(non_empty_positions) < 2:
        continue
    y_pos = non_empty_positions[-1]
    y_val = float(tokens[y_pos])

    # Numeric x values between function name and y
    x_vals = [tokens[i] for i in range(1, y_pos) if tokens[i] != ""]
    x_vals = [float(v) for v in x_vals]

    expected_dim = FUNCTION_DIMS[func]
    if len(x_vals) != expected_dim:
        print(f"Warning: line {line_no} for {func} has {len(x_vals)} x-values; expected {expected_dim}")

    x_padded = x_vals[:8] + [pd.NA] * (8 - len(x_vals))
    cleaned_records.append([func, *x_padded[:8], y_val, line_no])

df = pd.DataFrame(cleaned_records, columns=EXPECTED_COLUMNS + ["source_line"])
df.head(10)


,function,x1,x2,x3,x4,x5,x6,x7,x8,y,source_line
0,function_1,0.319404,0.762959,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1.322677e-79,3
1,function_1,0.574329,0.879898,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1.033078e-46,4
2,function_1,0.731024,0.733000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,7.710875e-16,5
3,function_1,0.840353,0.264732,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,3.341771e-124,6
4,function_1,0.650114,0.681526,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,-3.606063e-03,7
5,function_1,0.410437,0.147554,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,-2.159249e-54,8
6,function_1,0.312691,0.078723,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,-2.089093e-91,9
7,function_1,0.683418,0.861057,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2.535001e-40,10
8,function_1,0.082507,0.403488,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,3.606771e-81,11
9,function_1,0.883890,0.582254,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,6.229856e-48,12


In [4]:

# Tabulate the cleaned data.
print("Cleaned data shape:", df.shape)
display(df)


Cleaned data shape: (239, 11)


,function,x1,x2,x3,x4,x5,x6,x7,x8,y,source_line
0,function_1,0.319404,0.762959,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1.322677e-79,3
1,function_1,0.574329,0.879898,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1.033078e-46,4
2,function_1,0.731024,0.733000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,7.710875e-16,5
3,function_1,0.840353,0.264732,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,3.341771e-124,6
4,function_1,0.650114,0.681526,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,-3.606063e-03,7
...,...,...,...,...,...,...,...,...,...,...,...
234,function_4,0.482220,0.494750,0.425221,0.444092,<NA>,<NA>,<NA>,<NA>,-2.026346e+00,246
235,function_5,0.739843,0.999999,0.8783,0.996571,<NA>,<NA>,<NA>,<NA>,4.073099e+03,247
236,function_6,0.475087,0.419633,0.776163,0.645189,0.210294,<NA>,<NA>,<NA>,-4.067353e-01,248
237,function_7,0.186681,0.011299,0.464236,0.47268,0.468469,0.946441,<NA>,<NA>,7.703616e-01,250


In [5]:

# Count rows per function and compare with the initial counts.
counts = df.groupby("function").size().rename("row_count").reset_index()
counts["initial_count"] = counts["function"].map(INITIAL_COUNTS)
counts["additional_rows"] = counts["row_count"] - counts["initial_count"]
display(counts)


,function,row_count,initial_count,additional_rows
0,function_1,18,12,6
1,function_2,18,12,6
2,function_3,23,17,6
3,function_4,38,32,6
4,function_5,28,22,6
5,function_6,28,22,6
6,function_7,38,32,6
7,function_8,48,42,6


In [6]:

# Check whether the number of additional rows after the initial data is equal for all functions.
after_initial = counts[["function", "additional_rows"]].copy()
equal_after_initial = after_initial["additional_rows"].nunique() == 1
print("Equal number of rows added after the initial data for all functions:", equal_after_initial)
display(after_initial)


Equal number of rows added after the initial data for all functions: True


,function,additional_rows
0,function_1,6
1,function_2,6
2,function_3,6
3,function_4,6
4,function_5,6
5,function_6,6
6,function_7,6
7,function_8,6


In [7]:

    # Plot row number vs y for each function and mark the maximum y.
    saved_files = []

    for func, group in df.groupby("function", sort=True):
        group = group.reset_index(drop=True).copy()
        group["row_num"] = group.index + 1

        max_idx = group["y"].idxmax()
        max_row = group.loc[max_idx]

        plt.figure(figsize=(9, 5))
        plt.plot(group["row_num"], group["y"], marker="o")
        plt.scatter([max_row["row_num"]], [max_row["y"]], s=100)
        plt.annotate(f"max y = {max_row['y']:.6g} row = {int(max_row['row_num'])}",
            (max_row["row_num"], max_row["y"]),
            textcoords="offset points",
            xytext=(10, 10),
        )
        
        print(f"{func}: max y = {max_row['y']:.6g}")
        
        plt.title(f"{func}: row_num vs y")
        plt.xlabel("row_num")
        plt.ylabel("y")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()

        output_file = OUTPUT_DIR / f"{func}_rownum_vs_y.png"
        plt.savefig(output_file, dpi=150)
        plt.close()
        saved_files.append(str(output_file))

    print()
    print("Saved plot files:")
    for f in saved_files:
        print(" -", f)


function_1: max y = 0.000125546
function_2: max y = 0.663146
function_3: max y = -0.0348353
function_4: max y = -0.853868
function_5: max y = 4695.23
function_6: max y = -0.297123
function_7: max y = 1.36497
function_8: max y = 9.81032
Saved plot files:
 - plots/function_1_rownum_vs_y.png
 - plots/function_2_rownum_vs_y.png
 - plots/function_3_rownum_vs_y.png
 - plots/function_4_rownum_vs_y.png
 - plots/function_5_rownum_vs_y.png
 - plots/function_6_rownum_vs_y.png
 - plots/function_7_rownum_vs_y.png
 - plots/function_8_rownum_vs_y.png


In [8]:

# Optional: save the cleaned table for downstream use.
cleaned_csv = Path("cleaned_data.csv")
df.to_csv(cleaned_csv, index=False)
print("Saved cleaned CSV to:", cleaned_csv.resolve())


Saved cleaned CSV to: /Users/divine/Documents/Ashbrook Consulting Ltd/Certifications/ML & AI/Capstone project/Code/W9/cleaned_data.csv
